In [1]:
import pandas as pd
import os
from pathlib import Path

def analyze_bea_file_structure(file_path, max_rows=15):
    """BEA 파일의 구조를 상세 분석"""
    
    print(f"\n🔍 === {Path(file_path).name} 분석 ===")
    
    if not os.path.exists(file_path):
        print(f"❌ 파일 없음: {file_path}")
        return None
    
    try:
        # 여러 헤더 행으로 시도해보기
        for header_row in [0, 1, 2, 3, 4, 5, None]:
            print(f"\n📋 Header={header_row}로 읽기 시도:")
            
            try:
                if file_path.endswith('.csv'):
                    df = pd.read_csv(file_path, header=header_row, nrows=max_rows)
                else:
                    df = pd.read_excel(file_path, header=header_row, nrows=max_rows)
                
                print(f"   ✅ 성공! Shape: {df.shape}")
                print(f"   📊 컬럼들: {list(df.columns[:10])}")
                
                # 샘플 데이터 출력
                print(f"   📄 샘플 데이터:")
                for i in range(min(3, len(df))):
                    row_data = []
                    for j, col in enumerate(df.columns[:6]):  # 처음 6개 컬럼만
                        value = df.iloc[i, j]
                        if pd.isna(value):
                            row_data.append("NaN")
                        else:
                            row_data.append(str(value)[:15])  # 15자로 제한
                    print(f"      Row {i}: {row_data}")
                
                # 시계열 컬럼 후보 찾기
                time_candidates = []
                for col in df.columns:
                    col_str = str(col)
                    # 연도 패턴
                    if any(year in col_str for year in ['2020', '2021', '2022', '2023', '2024']):
                        time_candidates.append(col_str)
                    # 분기 패턴
                    elif 'Q' in col_str.upper() and any(q in col_str for q in ['1', '2', '3', '4']):
                        time_candidates.append(col_str)
                    # 숫자 컬럼
                    elif col_str.replace('.', '').replace('-', '').isdigit():
                        time_candidates.append(col_str)
                
                if time_candidates:
                    print(f"   🎯 시계열 후보: {time_candidates[:5]}")
                else:
                    print(f"   ⚠️ 시계열 컬럼 후보 없음")
                
                print("-" * 50)
                
            except Exception as e:
                print(f"   ❌ 실패: {str(e)[:100]}")
                
    except Exception as e:
        print(f"❌ 전체 분석 실패: {e}")
        return None

def analyze_all_bea_files():
    """모든 BEA 파일 분석"""
    
    FILES6 = {
        "RVA": "/Users/jiminbyun/Downloads/Real Value Added by Industry.xlsx",
        "RII": "/Users/jiminbyun/Downloads/Real Intermediate Input by Industry.xlsx", 
        "PII": "/Users/jiminbyun/Downloads/Chain-Type Price Indexes for Intermediate Inputs by Industry.xlsx",
        "RGO": "/Users/jiminbyun/Downloads/Real Gross Output by Industry.xlsx",
        "PGO": "/Users/jiminbyun/Downloads/Chain-Type Price Indexes for Gross Output by Industry.xlsx",
        "GO" : "/Users/jiminbyun/Downloads/Gross Output by Industry.xlsx",
    }
    
    print("🚀 모든 BEA 파일 구조 분석 시작...")
    
    results = {}
    
    for prefix, file_path in FILES6.items():
        print(f"\n{'='*60}")
        result = analyze_bea_file_structure(file_path)
        results[prefix] = result
    
    # 요약
    print(f"\n === 분석 요약 ===")
    for prefix, result in results.items():
        if result is None:
            print(f"❌ {prefix}: 분석 실패")
        else:
            print(f" {prefix}: 분석 완료")
    
    return results

def suggest_parsing_strategy():
    """파싱 전략 제안"""
    
    print(f"\n === BEA 파일 파싱 전략 제안 ===")
    print("1. 헤더 행 확인: 대부분 4-5번째 행이 실제 헤더")
    print("2. 산업명 컬럼: 보통 두 번째 컬럼 (Industry, Description 등)")
    print("3. 시계열 컬럼 패턴:")
    print("   - 연도: 2020, 2021, 2022, 2023, 2024")
    print("   - 분기: 2023Q1, 2023 Q1, Q1 2023")
    print("   - 숫자: 1, 2, 3, ... (순차적 컬럼)")
    print("4. 불필요한 행 제거: Addenda, Legend, Note, Source 포함")
    
    print(f"\n === 수정된 파싱 함수 제안 ===")
    
    parsing_code = '''
def improved_load_and_clean_bea(file_path: str, prefix: str):
    """개선된 BEA 파일 로더"""
    
    # 1. 여러 헤더로 시도
    for header_row in [4, 3, 5, 2, 1, 0]:
        try:
            if file_path.endswith('.csv'):
                df = pd.read_csv(file_path, header=header_row)
            else:
                df = pd.read_excel(file_path, header=header_row)
            
            # 2. 산업명 컬럼 찾기 (더 유연하게)
            industry_col = None
            for col in df.columns[:5]:  # 처음 5개 컬럼에서 찾기
                if df[col].dtype == 'object':
                    # 산업명이 많이 포함된 컬럼 찾기
                    sample_values = df[col].dropna().astype(str).head(10)
                    if any(keyword in ' '.join(sample_values).lower() 
                          for keyword in ['agriculture', 'manufacturing', 'mining', 'construction', 'retail']):
                        industry_col = col
                        break
            
            if industry_col is None:
                industry_col = df.columns[1]  # 기본값
            
            df.rename(columns={industry_col: "Industry"}, inplace=True)
            
            # 3. 시계열 컬럼 찾기 (더 포괄적)
            time_cols = []
            
            for col in df.columns:
                col_str = str(col).strip()
                
                # 연도 패턴 (2020-2024)
                if any(year in col_str for year in ['2020', '2021', '2022', '2023', '2024']):
                    time_cols.append(col)
                    
                # 분기 패턴
                elif ('Q' in col_str.upper() and 
                      any(q in col_str for q in ['1', '2', '3', '4'])):
                    time_cols.append(col)
                    
                # 숫자 컬럼 (연도일 가능성)
                elif (col_str.replace('.', '').isdigit() and 
                      len(col_str) >= 4):
                    time_cols.append(col)
                    
                # 숫자가 주로 들어있는 컬럼
                elif col != "Industry":
                    try:
                        numeric_ratio = pd.to_numeric(df[col], errors='coerce').notna().sum() / len(df)
                        if numeric_ratio > 0.7:  # 70% 이상이 숫자
                            time_cols.append(col)
                    except:
                        pass
            
            if time_cols:
                print(f"✅ {prefix}: {len(time_cols)}개 시계열 컬럼 발견 (헤더={header_row})")
                return df, time_cols
                
        except Exception as e:
            continue
    
    raise ValueError(f"{prefix}: 모든 헤더 시도 실패")
    '''
    
    print(parsing_code)

# 실행
if __name__ == "__main__":
    # 1. 모든 파일 분석
    results = analyze_all_bea_files()
    
    # 2. 파싱 전략 제안
    suggest_parsing_strategy()
    
    print(f"\n🎯 다음 단계:")
    print("1. 위 분석 결과를 보고 어떤 헤더 행이 적절한지 확인")
    print("2. 시계열 컬럼 후보들 중 실제 사용할 컬럼 선택")
    print("3. 개선된 파싱 함수로 코드 업데이트")

🚀 모든 BEA 파일 구조 분석 시작...


🔍 === Real Value Added by Industry.xlsx 분석 ===

📋 Header=0로 읽기 시도:
   ✅ 성공! Shape: (15, 11)
   📊 컬럼들: ['Real Value Added by Industry', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9']
   📄 샘플 데이터:
      Row 0: ['[Billions of 20', 'NaN', 'NaN', 'NaN', 'NaN', 'NaN']
      Row 1: ['Bureau of Econo', 'NaN', 'NaN', 'NaN', 'NaN', 'NaN']
      Row 2: ['Last Revised on', 'NaN', 'NaN', 'NaN', 'NaN', 'NaN']
   ⚠️ 시계열 컬럼 후보 없음
--------------------------------------------------

📋 Header=1로 읽기 시도:
   ✅ 성공! Shape: (15, 11)
   📊 컬럼들: ['[Billions of 2017 chain dollars] Seasonally adjusted at annual rates', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9']
   📄 샘플 데이터:
      Row 0: ['Bureau of Econo', 'NaN', 'NaN', 'NaN', 'NaN', 'NaN']
      Row 1: ['Last Revised on', 'NaN', 'NaN', 'NaN', 'NaN', 'NaN']
      Row 2: ['NaN', 'N

/Users/jiminbyun/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/jiminbyun/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/jiminbyun/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/jiminbyun/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/jiminbyun/anaconda3/lib/python3.11/site-packages/

   ✅ 성공! Shape: (15, 11)
   📊 컬럼들: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9']
   📄 샘플 데이터:
      Row 0: ['Line', 'NaN', '2023', 'NaN', 'NaN', 'NaN']
      Row 1: ['NaN', 'NaN', 'Q1', 'Q2', 'Q3', 'Q4']
      Row 2: ['1', '        Gross d', '22403.4', '22539.4', '22780.9', '22960.6']
   ⚠️ 시계열 컬럼 후보 없음
--------------------------------------------------

📋 Header=5로 읽기 시도:
   ✅ 성공! Shape: (15, 11)
   📊 컬럼들: ['Line', 'Unnamed: 1', '2023', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', '2024', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9']
   📄 샘플 데이터:
      Row 0: ['NaN', 'NaN', 'Q1', 'Q2', 'Q3', 'Q4']
      Row 1: ['1.0', '        Gross d', '22403.4', '22539.4', '22780.9', '22960.6']
      Row 2: ['2.0', 'Private industr', '19843.5', '19969.4', '20197.7', '20361.1']
   🎯 시계열 후보: ['2023', '2024', '2025']
--------------------------------------------------

📋 Header=None로 읽기 시도:
   ✅ 성공! Shape: (15, 11)
 

/Users/jiminbyun/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/jiminbyun/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/jiminbyun/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/jiminbyun/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
